# interpretable ML with SHAP




In [ ]:
from sklearn.cluster import AgglomerativeClustering

def drop_correlated_features( X , threshold = 0.9 ):
    """
    Args:
        - X (pd.DataFrame) : n,p feature matrix
        - threshold (float) : absolute correlation threshold group variables
    
    Returns:
        - pd.DataFrame : X with only the selected variables
        - dict : keys are the selected features , values are the list of features in the corresponding feature cluster 
    """
    
    corr_threshold = 0.9

    metric = 1 - X.corr().abs()

    HC = AgglomerativeClustering( n_clusters=None , metric='precomputed', linkage = 'single' , distance_threshold = (1-corr_threshold) )
    HC.fit(metric)

    variable_clusters = pd.Series( HC.labels_  , index = X.columns)

    cluster_to_features = variable_clusters.index.groupby(variable_clusters)

    ## keys are the selected feature in the cluster, values are the list of features in the cluster
    selected_features_to_features = { v[0]:list(v) for v in cluster_to_features.values() }

    return X.loc[:,selected_features_to_features.keys()] ,  selected_features_to_features 

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectPercentile
import numpy as np

## loading data
df_xpr = pd.read_csv("../data/TGCA_BRCA_expression_matrix.TPM.csv.gz" , index_col = 0)

df_clinical = pd.read_csv("../data/TGCA_BRCA_clinical_filtered.small.csv",index_col=0)
df_clinical = pd.get_dummies( df_clinical , drop_first=True)

## y is the poor_diagnosis
y = df_clinical.poor_prognosis

## ensuring the expression data is properly ordered
X_xpr = df_xpr.loc[ :, df_clinical.index].transpose() 

## selecting top 1% most variable genes
VT = SelectPercentile( score_func = lambda x,_ : np.var(x , axis = 0) ,
                       percentile = 1
                     )

X = pd.DataFrame( VT.fit_transform(X_xpr), columns=VT.get_feature_names_out() , index = X_xpr.index )
X , features_to_features_cluster = drop_correlated_features( X , threshold = 0.9 )


# adding age and sex to the set of features
X = pd.concat( [ df_clinical[['demographic.days_to_birth','demographic.sex_at_birth_male']] , X ] , axis=1 )



In [ ]:
## let's use the reduced set of features which we obtained from lasso in chapter 3
lasso_set = ["demographic.days_to_birth","ENSG00000102144.15","ENSG00000092010.15","ENSG00000178372.8","ENSG00000171209.3","ENSG00000115884.11","ENSG00000109971.14","ENSG00000175356.14","ENSG00000141367.12","ENSG00000101210.13","ENSG00000271503.6","ENSG00000133048.13","ENSG00000182054.10","ENSG00000124107.5","ENSG00000211640.4","ENSG00000143248.13","ENSG00000012660.14","ENSG00000080824.19","ENSG00000175426.11","ENSG00000138326.21","ENSG00000198888.2","ENSG00000205426.10","ENSG00000211648.2","ENSG00000129538.14","ENSG00000158710.15"]
X = X.loc[:,lasso_set]

X.shape

In [ ]:
y = y.values

## SHAP

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier


rf = RandomForestClassifier(n_jobs=-1, ccp_alpha = 10**-2.3, n_estimators=250 )
print( f"random forest cross-val accuracy: {cross_val_score(rf,X,y).mean():.3f}" )
_ = rf.fit(X,y)

## SHAP basic usage and plots

In [ ]:
import shap

%time explainer = shap.Explainer( rf ) ## creates an explainer from our model
%time shap_values = explainer(X) ## compute shap values for the prediction of the models on some data

In [ ]:
## explainer model is actually a subclass dedicated to trees
explainer

In [ ]:
## all shap values for all features and samples
shap_values

In [ ]:
## shap values for a specific feature
shap_values[:,'demographic.days_to_birth']

In [ ]:
shap_values[:,'demographic.days_to_birth'].values.shape

There is one column per class 

In [ ]:
y[:5]

In [ ]:
res_df = pd.DataFrame( {'pred':rf.predict( X ) , "truth":y  } )
res_df['prediction_ok'] = res_df.pred == res_df.truth
res_df.prediction_ok.value_counts()

In [ ]:
print("5 correct predictions of positive category:")
print( np.flatnonzero( res_df.prediction_ok & res_df.truth )[:5] )
print("5 correct predictions of positive category:")
print( np.flatnonzero( res_df.prediction_ok & ~res_df.truth )[:5] )
print("5 wrong predictions")
print( np.flatnonzero( ~res_df.prediction_ok )[:5] )

In [ ]:
?shap.plots.waterfall

In [ ]:
## explaination of a single prediction for category 1
shap.plots.waterfall(shap_values[0,:,1])

In [ ]:
## explaination of a single prediction for category 1
shap.plots.waterfall(shap_values[4,:,1])

In [ ]:
## explaination of a single prediction for category 1
shap.plots.waterfall(shap_values[133,:,1])

In [ ]:
## average SHAP value over all predictions --> proxy of feature importance 
shap.plots.bar(shap_values[...,1])

In [ ]:
## beeswarm gives us a sense of the relationship between the feature values and the SHAP
shap.plots.beeswarm(shap_values[...,1])

In [ ]:
## plotting the SHAP value for each observed age
shap.plots.scatter( shap_values[:,'demographic.days_to_birth',1] )

In [ ]:
shap.plots.scatter( shap_values[:,'ENSG00000102144.15',1] )

In [ ]:
shap.plots.scatter( shap_values[:,'ENSG00000109971.14',1] )

In [ ]:
## visualize some of the interaction between feature by coloring according to another variable 
shap.plots.scatter( shap_values[:,'ENSG00000109971.14',1] , color=shap_values[...,1] )

In [ ]:
## If you give all shap_values, the library searched for the one which may interact the most
shap.plots.scatter( shap_values[:,'ENSG00000109971.14',1] , color=shap_values[:,'demographic.days_to_birth',1] )

## SHAP interaction values

In [ ]:
%%time
shap_interaction_values = explainer.shap_interaction_values(X)

In [ ]:
import seaborn as sns
sns.heatmap( shap_interaction_values[1,...,1], 
            xticklabels = X.columns,
            yticklabels = X.columns,
           center = 0, cmap = 'bwr')

In [ ]:
shap_values[1,:,1].values

In [ ]:
shap_interaction_values[1,...,1].sum(axis = 1)

In [ ]:
clustering = shap.utils.hclust(X, y)
shap.plots.bar(shap_values[...,1], clustering=clustering,clustering_cutoff=1.0)

### exercise

We train a Logistic regression instead.

Look at the corresponding SHAP values. What do you see?


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


Xs = pd.DataFrame( StandardScaler().fit_transform(X),
                  columns = X.columns )
LR = LogisticRegression(C=np.inf)


print( f"logistic regression cross-val accuracy: {cross_val_score( LR, Xs,y).mean():.3f}" )
_ = LR.fit(Xs,y)

In [ ]:
explainer2 = shap.Explainer( LR , Xs ) ## creates an explainer from our model
shap_values2 = explainer2( Xs ) ## compute shap values for the prediction of the models on some data

In [ ]:
shap.plots.beeswarm(shap_values2)

In [ ]:
shap.plots.scatter( shap_values2[: , "demographic.days_to_birth"] )

### different explainer types

In [ ]:
%%time
explainer = shap.Explainer( rf ) ## defaults to TreeExplainer
print(type(explainer))
# TreeExplainer: 
#  * explores the tree ensemble to compute the SHAP values efficiently
#  * somewhat built-in the trees libraries nowadays -> extra fast computation
shap_values = explainer(X) 


In [ ]:
%%time

explainer = shap.Explainer( LR , Xs ) 
shap_values = explainer(Xs) 

explainer

In [ ]:
%%time
explainer = shap.Explainer( LR.predict_proba , Xs ) 
shap_values = explainer(Xs) 

explainer

[permutation explainer](https://shap.readthedocs.io/en/latest/example_notebooks/api_examples/explainers/Permutation.html)
 * uses smart permutation scheme to estimate SHAP values
 * needs a "background" sample which is used to "simulate" values for the masked features

An alternative is to explicitely say that it is a Linear model:

In [ ]:
%%time
# explain the model's predictions using SHAP
explainer = shap.explainers.Linear(LR, Xs)
shap_values = explainer(Xs)

In [ ]:
explainer

But in some case you won't have a choice and you need to go for a "generic" explainer such as PermutationExplainer

For example, for a KNN:

In [ ]:
%%time 
from sklearn.neighbors import KNeighborsClassifier

KNN = KNeighborsClassifier(n_neighbors=10)
KNN.fit(Xs, y)


explainer = shap.Explainer( KNN.predict_proba , Xs) 
shap_values = explainer(Xs) 

See the API documentation for [other "explainers"](https://shap.readthedocs.io/en/latest/api.html#explainers), either generic or adapted to specific kind of models.


### causation, correlation, and interpretation of predictive models

We are sure you have heard it a large number of time, but this message really is really worth repeating.

The SHAP documentation actually proposes an [insightful article](https://shap.readthedocs.io/en/latest/example_notebooks/overviews/Be%20careful%20when%20interpreting%20predictive%20models%20in%20search%20of%20causal%20insights.html) on the topic of misleading interpretation.

Our advise is to read it attentively, meditate on the subject, and then read it again.

